<a href="https://colab.research.google.com/github/Emimo56/GenAI-E1-BootCamp/blob/main/RAG_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q langchain-google-genai

In [9]:
!pip install -q langchain langchain-google-genai google-generativeai

In [10]:
# Import necessary packages
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Get the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

In [11]:
# Example interaction
response = llm.invoke("Write a short, catchy slogan for a coffee shop that specializes in unique coffee blends.")
print(response.content)

Here are a few options, playing on different angles:

**Short & Punchy:**

1.  **Beyond the Bean.**
2.  **Your Unique Cup.**
3.  **Taste the Unseen.**
4.  **Blend Your World.**

**Descriptive & Evocative:**

5.  **Discover Your Next Blend.**
6.  **The Art of the Blend.**
7.  **Uniquely Blended. Uniquely Yours.**
8.  **Your Daily Coffee Adventure.**


# **Langchain Ingestion and Retrieval Workflow**
We will now set up a Retrieval-Augmented Generation (RAG) workflow.
This involves ingesting a document into a vector store and then using an LLM to answer questions based on the retrieved information.

In [12]:
# Install additional necessary packages
%pip install -U chromadb tiktoken

In [13]:
# Install necessary packages for document loading if not already installed
%pip install -U langchain-community

# Define a dummy HR Policy Document
hr_policy_text = """# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company property. Harassment of any kind is strictly prohibited and will lead to disciplinary action, up to and including termination.

## Section 2: Leave Policy

### 2.1 Annual Leave

All full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

### 2.2 Sick Leave

Employees are allocated 10 days of paid sick leave per year. For absences exceeding three consecutive days, a doctor's note is required. All sick leave must be reported to the manager and HR within 24 hours of absence.

## Section 3: Expense Reimbursement

Employees can claim reimbursement for business-related expenses. Original receipts must be attached to all expense reports. Expenses must be submitted within 30 days of incurring them. The company will not reimburse personal expenses or expenses without valid receipts.

## Section 4: Remote Work Policy

Remote work arrangements are subject to manager approval and depend on job role suitability. Employees working remotely must maintain a secure and productive work environment. The company provides a stipend for essential home office equipment, upon request and approval.

## Section 5: Performance Reviews

Performance reviews are conducted semi-annually, in June and December, to assess employee performance and provide feedback for growth and development. Employees will have the opportunity to discuss their performance with their managers and set future goals."""

# Load the document using Langchain's TextLoader
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# Create a Document object directly from the string
documents = [Document(page_content=hr_policy_text, metadata={"source": "HR Policy Handbook"})]

print(f"Loaded {len(documents)} document.")
print("First 200 characters of the document:")
print(documents[0].page_content[:200])

Loaded 1 document.
First 200 characters of the document:
# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company 


**Split the Document into Chunks**
Large documents need to be split into smaller, manageable chunks so that the embeddings are more granular and relevant information can be retrieved effectively.

In [14]:
%pip install -U langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Max size of each chunk
    chunk_overlap=200, # Overlap between chunks to maintain context
    length_function=len,
    add_start_index=True,
)

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks.")
print("First chunk content:")
print(chunks[0].page_content)

Split document into 2 chunks.
First chunk content:
# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company property. Harassment of any kind is strictly prohibited and will lead to disciplinary action, up to and including termination.

## Section 2: Leave Policy

### 2.1 Annual Leave

All full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

### 2.2 Sick Leave

Employees are allocated 10 days of paid sick leave per year. For absences exceeding three consecutive days, a doctor's note is required. All sick leave must be reported to the manager and HR within 24 hours of absence.

## Section 3: Expense Reimburs

**Initialize Google Generative AI Embeddings**
We need an embedding model to convert our text chunks into numerical vectors. These vectors are then stored in the vector database and used for similarity search.

In [15]:

from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the embedding model
# Make sure GOOGLE_API_KEY is available from previous cells
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GOOGLE_API_KEY)

print("Google Generative AI Embeddings initialized.")

Google Generative AI Embeddings initialized.


**Create and Populate Chroma Vector Store**
Now, we'll use Chroma to store our document chunks and their corresponding embeddings. Chroma will enable us to quickly find relevant document parts when a query is made.

In [16]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Get the API key (ensure it's available for this cell's execution)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Re-initialize the embedding model with 'models/embedding-001'
# Both 'models/embedding-001' and 'text-embedding-004' are failing with NOT_FOUND errors.
# Reverting to 'models/embedding-001' as it was the model that initialized successfully in a prior cell.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2", google_api_key=GOOGLE_API_KEY)

# Create a Chroma vector store from the document chunks and embeddings
# We'll create an in-memory Chroma instance for this demonstration.
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Chroma vector store created with {vectorstore._collection.count()} entries.")

# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

print("Retriever created from Chroma vector store.")


Chroma vector store created with 6 entries.
Retriever created from Chroma vector store.


In [17]:
# The original %pip install command is removed to avoid dependency conflicts and is not needed for a functional RAG setup.
# The original imports for chain functions are removed as per the request to write code without chains.
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template for our LLM
# The context will be filled by the retrieved documents
prompt = ChatPromptTemplate.from_template(
    """Answer the user's question based on the provided context.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context: {context}
    Question: {input}"""
)

# Define a custom class to simulate the behavior of the retrieval_chain
# without using Langchain's chain objects.
class CustomRetrievalChain:
    def invoke(self, input_dict):
        # Assumes 'llm' and 'retriever' are available in the global scope
        # from previously executed cells.
        question = input_dict["input"]

        # 1. Retrieve relevant documents using the retriever
        retrieved_docs = retriever.invoke(question)

        # 2. Format the retrieved documents into a single context string
        #    Each document's page_content is joined by double newlines.
        formatted_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

        # 3. Format the prompt with the context and the user's question
        #    ChatPromptTemplate.format_messages returns a list of Message objects.


        formatted_prompt_messages = prompt.format_messages(
            context=formatted_context,
            input=question
        )

        # 4. Invoke the LLM with the formatted prompt messages
        response = llm.invoke(formatted_prompt_messages)

        # The result should match the structure of the original retrieval_chain's output.
        return {"answer": response.content}

# Instantiate the custom retrieval chain, so subsequent cells can call retrieval_chain.invoke()
retrieval_chain = CustomRetrievalChain()

print("RAG 'retrieval_chain' (manual implementation) created successfully.")


RAG 'retrieval_chain' (manual implementation) created successfully.


**Demonstrate Retrieval**
Now, let's ask a question about the HR policy and see how the RAG chain responds.

In [18]:
# Ask a question about the HR policy
question = "What is the policy for annual leave?"
print(f"\nQuestion: {question}")

# Invoke the RAG chain
response = retrieval_chain.invoke({"input": question})
print(f"\nAnswer: {response['answer']}")

question_2 = "Can I roll over unused sick leave?"
print(f"\nQuestion: {question_2}")
response_2 = retrieval_chain.invoke({"input": question_2})
print(f"\nAnswer: {response_2['answer']}")

question_3 = "How often are performance reviews conducted and what is their purpose?"
print(f"\nQuestion: {question_3}")
response_3 = retrieval_chain.invoke({"input": question_3})
print(f"\nAnswer: {response_3['answer']}")


Question: What is the policy for annual leave?

Answer: Full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

Question: Can I roll over unused sick leave?

Answer: I don't know. The policy handbook states that unused *annual* leave days do not roll over, but it does not specify whether unused *sick* leave can be rolled over.

Question: How often are performance reviews conducted and what is their purpose?

Answer: Performance reviews are conducted semi-annually, in June and December. Their purpose is to assess employee performance, provide feedback for growth and development, and offer employees the opportunity to discuss their performance with managers and set future goals.
